In [ ]:
import sys
import os

# add src folder to path
sys.path.append(os.path.abspath("../src")) 

from imports import * 

In [ ]:
''' 
This notebook performs prices forecasting using past observations: 
        Historical data is split into training (2014-2023) and test (2024) periods (done)
        A sARIMAX(p,d,q) is trained on the returns timeseries (done)
        Model results scored and recored (done)
        Forecast is plotted against true observations (done)

'''

In [ ]:
# load prices.csv
prices = pd.read_csv("../data/prices.csv", index_col=0, parse_dates=True)

print(prices.info())
prices

In [ ]:
# load returns.csv
returns = pd.read_csv("../data/returns.csv", index_col=0, parse_dates=True) 

print(returns.info())
returns

In [ ]:
# choose which data to use for modeling
# forecast_name = 'prices'
# data = prices.copy()

# forecast_name = 'returns'
# data=returns.copy() 

In [ ]:
tickers = data.columns
tickers

In [ ]:
# Split data into train, and test sets 
train_split = '2023-01-01' 

train = data.loc[:'2022-12-31'] 
test = data.loc[train_split:]

# check
print(train.shape, test.shape)
print(train.index.min(), train.index.max()) 
print(test.index.min(), test.index.max())

print("Train")
print(train) 
print("Test")
print(test)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(train.index, train, label="train") 
plt.plot(test.index, test, label="test")

plt.title("Data Split")
plt.legend()
plt.show()

In [ ]:
# A first model
model_dict = {}

if forecast_name == 'prices':
    p, d, q = (1, 1, 1) # p, d, q
else:
    p, d, q = (1, 0, 1) 

for ticker in tickers:
    # sARIMAX(p,d,q,s)
    model = SARIMAX( train[ticker], 
                        order=(p,d,q),
                        # trend='ct',
                        # time_varying_regression=True,
                        )
    # fit model
    model_history = model.fit()
    model_dict[ticker] = model_history

    print(f'\n  {ticker} \n')
    print(f'\n     sARIMAX({p},{d},{q}) Summary: \n')
    print(model_history.summary())

In [ ]:
# Single 1-step ahead forecast
preds = pd.DataFrame(index=test.index, columns=tickers)

for ticker in tickers:
    print(ticker) 
    model_results = model_dict[ticker]

    # 1-step variance forecast
    pred = model_results.forecast(steps=1)  

    print(f'Date: {test.index[0]}')    
    print(f'    Actual Prices 1-step ahead:     {test[ticker].iloc[0]:.4f}') 
    print(f'    Predicted Prices 1-step ahead:  {float(pred.iloc[0])}') 

In [ ]:
# Rolling forecast 1-step ahead 
preds = pd.DataFrame(index=test.index, columns=tickers)

window_size = 1

time_spent = {}
for ticker in tickers: 
    preds_list = [] 

    history = train[ticker].copy()

    model_results = model_dict[ticker]
    
    start = time.time()
    for i, h in enumerate(test.index): 
        if i % window_size == 0:
            model = SARIMAX(history, 
                    order=(p,d,q),
                    # trend='ct',
                    # time_varying_regression=True,
                    )
            model_results = model.fit(disp='off')
            
        # 1-step ahead forecast 
        pred = model_results.forecast(steps=1)
        
        # add to predictions
        preds_list.append(float(pred.iloc[0]))  

        # update history
        history.loc[h] = test[ticker].loc[h] 
        
    preds[ticker] = preds_list
    end = time.time() 
    time_spent[ticker] = end - start

for ticker in tickers:
    mins, secs = time_spent[ticker] // 60, time_spent[ticker] % 60
    print(ticker)
    print(f'Time elapsed: {int(mins)}m {secs}s')

preds

In [ ]:
# evaluate model
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return 100 * np.mean(2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scores = {}
for ticker in tickers:   
    
    rootmeansquarederr = rmse(test[ticker], preds[ticker])
    sMAPE = smape(test[ticker], preds[ticker]) 

    scores[ticker] = {'RMSE': rootmeansquarederr, 'sMAPE': sMAPE} 

scores

In [ ]:
# scores to data
scores_df = pd.DataFrame.from_dict(scores, orient='index') 

# save scores
scores_df.to_csv(f'../results/tables/sarima{p}{d}{q}_{forecast_name}_scores.csv')

scores_df

In [ ]:
for ticker in tickers:
    plt.figure(figsize=(12, 5))
    
    title = f"{ticker} Actual vs sARIMA({p},{d},{q}) {forecast_name.capitalize()} Forecast"

    test[ticker].plot(label="Observed")
    preds[ticker].plot(label=f"sARIMA({p},{d},{q}) Forecast")
    
    plt.title(title)

    # save plot 
    plt.savefig(f"../results/figures/{title}.png", dpi=300, bbox_inches="tight")

    plt.legend()
    plt.show()

In [ ]:
results = preds.copy()

new_columns = [] 
for ticker in tickers:
    new_columns.append(f'{ticker}_forecast')

results.columns = new_columns

print(results.info())
results

In [ ]:
# save results
results.to_csv(f"../results/forecasts/sARIMA{p}{d}{q}_{forecast_name}_forecasts.csv")